# UI-TARS-2B-SFT diagnostic

Why does the off-the-shelf model get ~27% type-acc on our AndroidControl harness
when the paper reports ~67% step accuracy?

This notebook walks through the knobs that could matter (system prompt,
coord scale, image preprocessing, prev_actions serialization), runs the model
under each variant on a single fixed test step, and prints the **raw model
output** + parsed action + match against ground truth.

Run cells top to bottom. The model loads once (cell 3) and is reused across
all variant cells.

## 1. Setup

In [ ]:
import sys, json, time
from pathlib import Path
from PIL import Image
import torch

HERE = Path.cwd()  # assumes you launch jupyter from Session2/ui-tars/
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent / '_act_ui'))

from harness import TogglableUITars, parse_action
from prompts import PROMPTS
from prepare_data import iter_steps_with_images

print('torch:', torch.__version__, ' mps:', torch.backends.mps.is_available())
print('prompt variants:', list(PROMPTS.keys()))

## 2. Pick the test step we'll probe

Default: episode 19349 step 2 — the `click(0.546, 0.077)` on the M&S app
search bar. This is a clean single-target click on a familiar app screen, so
any half-competent VLM should land it. Change `EPISODE_ID` / `STEP_IDX` to
stress-test other cases.

In [ ]:
EPISODE_ID = 19349
STEP_IDX   = 2

steps = list(iter_steps_with_images('test', ep_filter={EPISODE_ID}))
steps.sort(key=lambda s: s['step_id'])
step = steps[STEP_IDX]
img = Image.fromarray(step['img']).convert('RGB')

print('goal      :', step['instruction'])
print('GT action :', step['type_name'], step.get('xy'))
print('img size  :', img.size)
img.resize((300, int(300 * img.size[1] / img.size[0])))

## 3. Load the model once (~30 s on MPS)

In [ ]:
agent = TogglableUITars(prompt_variant='A_current',
                         coord_scale=1000.0,
                         prev_actions_mode='string',
                         max_side=896)
agent.load()
print('loaded.', agent.model_id, ' device:', agent.model.device)

## 4. Baseline — what we ship today

Prompt variant A, 0–1000 coord scale, string-serialized prev_actions, 896px max-side.

In [ ]:
out = agent.predict_step_raw(img, step['instruction'], prev_actions=[])
print('--- RAW ---')
print(out['raw'])
print('--- PARSED ---')
print(json.dumps(out['parsed'], indent=2))
print(f'inference time: {out["sec"]:.2f} s')

## 5. Ablate the system prompt

Same step, four different system prompts. Watch the raw output change.
If one variant produces a `click(start_box='(x,y)')` with coords near
`(546, 77)` (the 1000-scaled GT), that's our winner.

In [ ]:
results = []
for variant in PROMPTS:
    agent.prompt_variant = variant
    out = agent.predict_step_raw(img, step['instruction'], prev_actions=[])
    p = out['parsed']
    pred_str = p['action_type']
    if 'x' in p.get('params', {}):
        pred_str += f"({p['params']['x']:.3f}, {p['params']['y']:.3f})"
    print(f'== {variant} ==')
    print('  pred:', pred_str)
    print('  raw :', out['raw'][:250].replace('\n', ' ⏎ '))
    print()
    results.append({'variant': variant, **out})

## 6. Ablate the image-size cap

AndroidControl screenshots are 1080×2400 (tall). Our default crushes them
to a 896-pixel max-side. Try without.

In [ ]:
agent.prompt_variant = 'B_official_box_markers'  # use best prompt from §5
for ms in (896, 1280, None):
    agent.max_side = ms
    out = agent.predict_step_raw(img, step['instruction'], prev_actions=[])
    p = out['parsed']
    pred_str = p['action_type']
    if 'x' in p.get('params', {}):
        pred_str += f"({p['params']['x']:.3f}, {p['params']['y']:.3f})"
    print(f'max_side={ms!s:<5s}  img={out["img_size"]}  → {pred_str}')

## 7. Ablate prev_actions serialization

For a multi-step episode, does feeding past actions as a single string
vs. as alternating chat turns vs. omitting them entirely change behavior?

We'll feed the first 2 ground-truth actions of episode 19349 as `prev_actions`
and probe step 2 again.

In [ ]:
prev = ['open_app(M&S)', 'wait()']
agent.prompt_variant = 'B_official_box_markers'
agent.max_side = 896
for mode in ('none', 'string', 'chat_history'):
    agent.prev_actions_mode = mode
    out = agent.predict_step_raw(img, step['instruction'], prev_actions=prev)
    p = out['parsed']
    pred_str = p['action_type']
    if 'x' in p.get('params', {}):
        pred_str += f"({p['params']['x']:.3f}, {p['params']['y']:.3f})"
    print(f'prev_actions_mode={mode:<14s} → {pred_str}')

## 8. Putting it together: best config so far

Run the configuration that won the ablations on a full episode end-to-end
and compare to ACT-small / ACT-large from `_act_ui/reports/`.

In [ ]:
BEST_PROMPT  = 'B_official_box_markers'   # ← update after running §5
BEST_MAX_SIDE = 896                        # ← update after running §6
BEST_PREV    = 'chat_history'              # ← update after running §7

agent.prompt_variant   = BEST_PROMPT
agent.max_side         = BEST_MAX_SIDE
agent.prev_actions_mode = BEST_PREV

ep_results = []
prev = []
for s in steps:
    pil = Image.fromarray(s['img']).convert('RGB')
    out = agent.predict_step_raw(pil, s['instruction'], prev_actions=prev)
    p = out['parsed']
    type_ok = (p['action_type'] == s['type_name'])
    ep_results.append({'step': s['step_id'], 'gt': s['type_name'],
                       'pred': p['action_type'], 'ok': type_ok,
                       'params': p.get('params')})
    prev.append(f"{p['action_type']}({p.get('params', {})})")

import pandas as pd
pd.DataFrame(ep_results)

## 9. Port the winning config back into the main harness

Once §5–§8 give you a configuration that scores at least ~50% type-acc on
this episode, edit `Session2/_act_ui/infer.py::UITarsAgent`:

* replace `UITARS_SYSTEM` with `PROMPTS['<winning variant>']`
* set `prev_actions_mode` accordingly in `predict_step`
* adjust `MAX_SIDE` if a different value won

Then rerun `_act_ui/demo_episodes.py --agents uitars --n_episodes 2` and
rebuild the deck with `build_one_deck.py` — UI-TARS will reappear with real
numbers on the eval slide and meaningful predictions in the per-step slides.